##PROTOTYPE

In [ ]:

!pip install gTTS

from gtts import gTTS
from IPython.display import Audio, display

def speak(text, lang='th'):
    tts = gTTS(text=text, lang=lang)
    tts.save("speech.mp3")
    display(Audio("speech.mp3", autoplay=True))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:

import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, Audio
from gtts import gTTS
import random
from sklearn.ensemble import RandomForestClassifier

bts_data = pd.DataFrame({
    "station": ["A", "B", "C", "D", "E"],
    "lift": [1, 1, 0, 1, 0],
    "ramp": [1, 1, 1, 0, 0],
    "safe_area": [1, 1, 1, 1, 0]
})

def label_station(row):
    score = row["lift"]*2 + row["ramp"] + row["safe_area"]
    return 1 if score >= 3 else 0

bts_data["accessible"] = bts_data.apply(label_station, axis=1)

X_bts = bts_data[["lift", "ramp", "safe_area"]]
y_bts = bts_data["accessible"]

bts_model = DecisionTreeClassifier(max_depth=3)
bts_model.fit(X_bts, y_bts)


data = pd.DataFrame({
    "transport": ["BTS", "MRT", "Bus", "Grab"],
    "has_lift": [1, 1, 0, 0],
    "ramp": [1, 1, 1, 0],
    "safe": [1, 1, 1, 1],
    "cost": [40, 45, 20, 130],
    "name": ["BTS", "MRT", "Thai Smile Bus", "Grab"],
    "recommended": ["yes", "no", "yes", "yes"]
})


le = LabelEncoder()
data["recommended"] = le.fit_transform(data["recommended"])

X = data[["has_lift", "ramp", "safe", "cost"]]
y = data["recommended"]

model = RandomForestClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42
)

model.fit(X, y)


usage_widget = widgets.ToggleButtons(
    options=['ไม่บ่อย', 'บ่อย'],
    description='ความถี่:'
)

rush_widget = widgets.ToggleButtons(
    options=['ไม่รีบ', 'รีบมาก'],
    description='ความเร่งด่วน:'
)

feedback_widget = widgets.SelectMultiple(
    options=['เดินทางช้า', 'ค่าใช้จ่ายสูง', 'ไม่สะดวก','ลิฟต์เสีย', 'ทางลาดชันเกินไป',],
    description='Feedback:'
)

run_button = widgets.Button(description="ให้ AI แนะนำ ")


user_history = {"high": ["BTS", "BTS", "Bus"]}
feedback_history = {}

def get_behavior_preference(user_key):
    if user_key in user_history:
        return Counter(user_history[user_key]).most_common(1)[0][0]
    return None


def speak(text, lang='th'):
    tts = gTTS(text=text, lang=lang)
    tts.save("speech.mp3")
    display(Audio("speech.mp3", autoplay=True))


def run_ai(b):
    usage = usage_widget.value
    rush = rush_widget.value

    def calculate_score(row):
        score = 0

        sample = pd.DataFrame([{
            "lift": row["has_lift"],
            "ramp": row["ramp"],
            "safe_area": 1
        }])

        ai_access = bts_model.predict(sample)[0]


        if ai_access == 1:
            score += 4
        else:
            score -= 2


        score += 3 if row["ramp"] == 1 else -2
        score += 3 if row["safe"] == 1 else 0
        score += 2 if row["cost"] < 50 else 0

        if usage == "high":
            score += 1

        if rush == "รีบมาก" and row["transport"] == "Grab":
            score += 3


        score += random.uniform(-1, 1)

        return score

    data["score"] = data.apply(calculate_score, axis=1)
    data["ai_prediction"] = model.predict(X)


    preferred = get_behavior_preference(usage)
    if preferred:
        data.loc[data["transport"] == preferred, "score"] += 3


    recommended = data[data["ai_prediction"] == 1].sort_values("score", ascending=False)

    option1 = recommended.iloc[0]
    option2_3 = recommended.iloc[1:3]


    print("\n===== AI Recommendation =====")

    print("\n 🥇ทางเลือกที่1:", option1["name"])
    print("รายละเอียด:",
          f"ลิฟต์={option1['has_lift']}, ramp={option1['ramp']}, ราคา={option1['cost']}")

    print("\n เหตุผล:")
    if option1["has_lift"]:
        print("- มีลิฟต์ รองรับ wheelchair")
    if option1["ramp"]:
        print("- มีทางลาด")
    if option1["cost"] < 50:
        print("- ค่าใช้จ่ายประหยัด")
    if rush == "รีบมาก" and option1["transport"] == "Grab":
        print("- เหมาะกับกรณีเร่งด่วน")

    print("\n 🥈ทางเลือกอื่น:")
    for _, row in option2_3.iterrows():
        print("-", row["name"], f"(score={round(row['score'],2)})")


    speech = f"แนะนำตัวเลือกหลัก {option1['name']}"
    speak(speech)


    feedback = feedback_widget.value
    if feedback:
        for t in [option1["transport"]]:
            for reason in feedback:
                if reason == "เดินทางช้า":
                    data.loc[data["transport"] == t, "score"] -= 2
                elif reason == "ค่าใช้จ่ายสูง":
                    data.loc[data["transport"] == t, "score"] -= 2
                elif reason == "ไม่สะดวก":
                    data.loc[data["transport"] == t, "score"] -= 1
                elif reason == "ลิฟต์เสีย":
                    data.loc[data["transport"] == t, "score"] -= 3
                elif reason == "ทางลาดชันเกินไป":
                    data.loc[data["transport"] == t, "score"] -= 2

        print("\n[AI] เรียนรู้จาก Feedback แล้ว ✅")


run_button.on_click(run_ai)


display(usage_widget, rush_widget, feedback_widget, run_button)

ToggleButtons(description='ความถี่:', options=('ไม่บ่อย', 'บ่อย'), value='ไม่บ่อย')

ToggleButtons(description='ความเร่งด่วน:', options=('ไม่รีบ', 'รีบมาก'), value='ไม่รีบ')

SelectMultiple(description='Feedback:', options=('เดินทางช้า', 'ค่าใช้จ่ายสูง', 'ไม่สะดวก', 'ลิฟต์เสีย', 'ทางล…

Button(description='ให้ AI แนะนำ ', style=ButtonStyle())


===== AI Recommendation =====

 🥇ทางเลือกที่1: BTS
รายละเอียด: ลิฟต์=1, ramp=1, ราคา=40

 เหตุผล:
- มีลิฟต์ รองรับ wheelchair
- มีทางลาด
- ค่าใช้จ่ายประหยัด

 🥈ทางเลือกอื่น:
- Thai Smile Bus (score=5.54)
- Grab (score=-0.36)



[AI] เรียนรู้จาก Feedback แล้ว ✅
